# CS383: Data Science and Machine Learning
## Lecture 4 — Datetime Handling, Data Cleaning, and Reshaping

*Dr. Thitima Srivatanakul*

### Guiding question
**Once you have real data in Pandas, what actually needs fixing before you can trust it?**

### Learning objectives
By the end of this lecture, you should be able to:

- parse text into real datetime values and pull useful components out of them (year, month, day of week, hour);
- compute a duration between two datetime columns;
- check a dataset for missing values, duplicate rows, inconsistent text, and wrong data types;
- reshape a table between wide and long form with `pivot_table` and `melt`;
- combine multiple batches of data with `pd.concat`.

---

### Where this fits
Lecture 3 (and its in-class joins exercise) deliberately left `created_date` alone and kept the data tidy. Today we stop avoiding both of those things: real datetime values, and real mess. Everything here uses the same familiar NYC 311 dataset, on purpose — Assignment 1 will ask you to apply these same skills to a dataset you've never seen before.

---
**Live in-class version.** Type along at each `__________` blank — everything else is filled in so class time stays on the new syntax, not on retyping boilerplate.

---

## Part 1 — Datetime Handling

So far, `created_date` has just been text that happened to look like a date. Today it actually becomes one — and this time we're also pulling `closed_date`, so we can measure how long complaints take to resolve.

### Setup

Same live-pull-with-fallback pattern as before, now including `closed_date`. A few rows are deliberately messed up on purpose (some duplicated, some with inconsistent borough capitalization) — Part 2 needs real problems to practice on, and this guarantees there are some no matter what the network looks like today.

In [1]:
import sqlite3
import numpy as np
import pandas as pd
import requests

SOCRATA_URL = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"

try:
    response = requests.get(
        SOCRATA_URL,
        params={
            "$limit": 8000,
            "$order": "created_date DESC",
            "$select": "unique_key,complaint_type,borough,created_date,closed_date",
        },
        timeout=8,
    )
    response.raise_for_status()
    complaints_df = pd.DataFrame(response.json())
    live = True
except Exception:
    # Offline fallback, in case there is no internet connection in the room.
    rng = np.random.default_rng(383)
    n = 8000
    complaint_types = ["Noise - Residential", "Illegal Parking", "HEAT/HOT WATER",
                        "Blocked Driveway", "Street Condition", "Water System",
                        "PAINT/PLASTER", "Damaged Tree", "Sewer", "Rodent"]
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]

    created = pd.date_range("2026-01-01", periods=n, freq="min")
    still_open = rng.random(n) < 0.15  # about 15% of complaints have no closed_date yet
    resolution_hours = rng.gamma(shape=2.0, scale=20.0, size=n)
    closed = created + pd.to_timedelta(resolution_hours, unit="h")
    closed_str = np.where(still_open, None, closed.astype(str))

    complaints_df = pd.DataFrame({
        "unique_key": np.arange(1, n + 1).astype(str),  # Socrata's API returns everything as text
        "complaint_type": rng.choice(
            complaint_types, size=n,
            p=[0.18, 0.15, 0.14, 0.10, 0.10, 0.09, 0.08, 0.06, 0.05, 0.05],
        ),
        "borough": rng.choice(boroughs_list, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "created_date": created.astype(str),
        "closed_date": closed_str,
    })
    live = False

# Deliberately introduce some real-world messiness, so there's something to clean
# in Part 2 regardless of whether today's data is live or offline.
rng2 = np.random.default_rng(99)
messy_idx = rng2.choice(complaints_df.index, size=15, replace=False)
complaints_df.loc[messy_idx, "borough"] = complaints_df.loc[messy_idx, "borough"].str.lower()
complaints_df = pd.concat([complaints_df, complaints_df.sample(6, random_state=1)], ignore_index=True)

print(f"{'Live' if live else 'Offline fallback'} data: {len(complaints_df):,} 311 records")
complaints_df.head()

Live data: 8,006 311 records


,unique_key,complaint_type,borough,created_date,closed_date
0,70027941,Noise - Residential,Unspecified,2026-08-12T02:06:17.000,NaN
1,70026445,Noise - Vehicle,BRONX,2026-08-12T02:06:16.000,NaN
2,70032610,Noise - Street/Sidewalk,BROOKLYN,2026-08-12T02:05:38.000,NaN
3,70033308,Illegal Parking,QUEENS,2026-08-12T02:03:32.000,NaN
4,70037864,Illegal Parking,QUEENS,2026-08-12T02:03:24.000,NaN


### Parsing text into real datetimes

In [4]:
print("Before parsing:")
print(complaints_df[["created_date", "closed_date"]].dtypes)

complaints_df["created_date"] = pd.__________(complaints_df["created_date"])
complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])

print("\nAfter parsing:")
print(complaints_df[["created_date", "closed_date"]].dtypes)

Before parsing:
created_date    datetime64[ns]
closed_date     datetime64[ns]
dtype: object

After parsing:
created_date    datetime64[ns]
closed_date     datetime64[ns]
dtype: object


`pd.to_datetime()` converts a column of date-like text into an actual `datetime64` type. Once it's real, Pandas understands it as a date — not just a string that happens to look like one — which unlocks everything below.

### When parsing goes wrong

In [ ]:
messy_dates = pd.Series(["2026-01-15", "2026-02-30", "2026-03-01"])

try:
    pd.to_datetime(messy_dates)
except Exception as e:
    print(f"Parsing failed: {e}")

Real data isn't always clean — a source system might record `"2026-02-30"` (a date that doesn't exist) or use a format `pd.to_datetime()` can't figure out on its own. When that happens, `errors="coerce"` turns anything invalid into `NaT` (Pandas' missing-datetime marker) instead of crashing the whole column:

In [ ]:
pd.to_datetime(messy_dates, errors="__________")

You'll use this exact `errors="coerce"` pattern again below, when `unique_key` turns out to have the same kind of problem.

### Pulling components out of a datetime

In [ ]:
complaints_df["created_year"] = complaints_df["created_date"].dt.year
complaints_df["created_month"] = complaints_df["created_date"].dt.month
complaints_df["created_day_name"] = complaints_df["created_date"].dt.day_name()
complaints_df["created_hour"] = complaints_df["created_date"].dt.__________

complaints_df[["created_date", "created_year", "created_month", "created_day_name", "created_hour"]].head()

Every datetime column gets a `.dt` accessor, which unlocks these component fields. `.dt.day_name()` is a nice one for questions like "which day of the week gets the most complaints?"

### Computing a duration

In [ ]:
complaints_df["resolution_time_hours"] = (
    complaints_df["closed_date"] - complaints_df["created_date"]
).dt.__________() / 3600

complaints_df[["created_date", "closed_date", "resolution_time_hours"]].head(10)

Subtracting one datetime column from another gives you a `timedelta` — a duration. `.dt.total_seconds()` converts that into a plain number, which we then divide by 3600 to get hours. Look closely at a few rows: some show `NaT` for `closed_date` and `NaN` for `resolution_time_hours`. That's not a mistake — those are complaints that haven't been resolved yet, so there's no closing time to subtract. That's exactly where Part 2 picks up.

---

## Part 2 — Real-World Data Cleaning Patterns

Four checks worth running on almost any dataset before you trust it: missing values, duplicate rows, inconsistent text, and wrong data types.

### Checking for missing values

In [ ]:
complaints_df.__________().sum()

`isna()` marks every missing value as `True`, and `.sum()` (treating `True` as 1) counts them per column. Notice `closed_date` and `resolution_time_hours` are the only columns with real gaps — and we already know why: those are the still-open complaints from Part 1.

This matters: that missingness isn't random noise to be smoothed over. An open complaint doesn't have a "typical" resolution time yet — it has *no* resolution time. Filling it in with an average would quietly invent a fact that isn't true. The better move is usually to leave it as missing and, if it's useful, add a separate flag instead:

In [ ]:
complaints_df["is_still_open"] = complaints_df["closed_date"].__________()
complaints_df["is_still_open"].value_counts()

### Checking for duplicate rows

In [ ]:
print(f"Duplicate rows: {complaints_df.duplicated().sum()}")

complaints_df = complaints_df.__________().reset_index(drop=True)
print(f"Rows after dropping duplicates: {len(complaints_df):,}")

`duplicated()` flags rows that are exact repeats of an earlier row; `drop_duplicates()` removes them, keeping the first occurrence by default.

### Checking for inconsistent text

In [ ]:
print(complaints_df["borough"].__________())

In [ ]:
complaints_df["borough"] = complaints_df["borough"].str.strip().str.__________()
print(complaints_df["borough"].unique())

A few rows had `"brooklyn"` or `" queens"` mixed in with the correctly-formatted `"BROOKLYN"` and `"QUEENS"`. To Pandas, those are entirely different categories until you normalize them — `.str.strip()` removes stray whitespace, `.str.upper()` fixes the casing.

### Checking data types

In [ ]:
print(complaints_df["unique_key"].dtype)
print(complaints_df["unique_key"].head())

In [ ]:
complaints_df["unique_key"] = pd.__________(complaints_df["unique_key"], errors="coerce")
print(complaints_df["unique_key"].dtype)

`unique_key` looks numeric, but APIs like this one return every field as text by default — so it often comes in as an `object` (string) column. `pd.to_numeric(..., errors="coerce")` converts it to a real numeric type, and turns anything that *can't* be converted into `NaN` instead of crashing — which is itself another way missing/bad values can quietly show up.

### A slow, easy-to-get-wrong way to update rows

In [ ]:
# Say you want to flag complaints as "High" or "Standard" priority based on type.
priority_types = {"HEAT/HOT WATER", "Water System", "Sewer"}

for index, row in complaints_df.iterrows():
    row["priority"] = "High" if row["complaint_type"] in priority_types else "Standard"

"priority" in complaints_df.columns

Spot the bug: nothing happened — `"priority" in complaints_df.columns` comes back `False`. `iterrows()` hands you a *copy* of each row, so `row["priority"] = ...` sets a value on that throwaway copy and never touches `complaints_df` itself. Even fixed (e.g. `complaints_df.loc[index, "priority"] = ...`), looping row-by-row over 8,000 rows repeats the same slow, un-vectorized work you saw with `np.vectorize()` in Lecture 2 and `.apply()` in Lecture 3.

In [ ]:
complaint_to_priority = {t: "High" for t in priority_types}
complaints_df["priority"] = complaints_df["complaint_type"].__________(complaint_to_priority).fillna("Standard")
complaints_df[["complaint_type", "priority"]].head()

`.map()` looks up each value in a dictionary and returns the match — any complaint type not in `complaint_to_priority` comes back as `NaN`, which `.fillna("Standard")` catches. This is the vectorized, dictionary-based way to do value-by-value replacement. `.replace()` does the same job with the same dictionary — worth recognizing either one in code you read.

You'll also see `fillna()` used to replace missing values with a placeholder — `fillna(0)`, or a column's mean or median — instead of leaving them as `NaN` or dropping them. We're deliberately not using it in this lecture: doing it safely, without leaking information from data your model shouldn't see yet, is something you'll learn properly in Week 6.

---

## Part 3 — Reshaping and Combining Multiple Sources

### Wide form, with pivot_table

In [ ]:
pivot = complaints_df.pivot_table(
    index="borough", columns="complaint_type", values="unique_key", aggfunc="__________", fill_value=0
)
pivot

This is the same shape of table `pd.crosstab()` gave you in Lecture 3 — one row per borough, one column per complaint type. `pivot_table` is the more general tool: it can aggregate any column with any function (`count`, `mean`, `sum`, ...), not just count rows.

### Back to long form, with melt

In [ ]:
long_form = pivot.reset_index().__________(
    id_vars="borough", var_name="complaint_type", value_name="count"
)
long_form.head()

`melt` undoes a pivot: instead of one column per complaint type, you get one row per (borough, complaint_type) combination. Wide form is often easier to *read*; long form is often easier to *compute or plot with* — which is why melt shows up so often right before charting.

### Combining multiple batches with concat

In [ ]:
# Imagine these arrived as two separate exports -- say, two different weeks' worth of data --
# that need to be stacked into one combined table.
batch_a = complaints_df.iloc[:4000]
batch_b = complaints_df.iloc[4000:]

combined = pd.__________([batch_a, batch_b], ignore_index=True)
print(f"batch_a: {len(batch_a):,} rows | batch_b: {len(batch_b):,} rows | combined: {len(combined):,} rows")

`pd.concat()` stacks tables with the *same columns* on top of each other — useful any time data arrives in separate pieces (multiple exports, multiple months, multiple files) that all share the same structure. This is different from the `.merge()` you used in the Lecture 3 joins exercise: `merge` combines tables *sideways*, matching rows by a shared key; `concat` combines tables *vertically*, just gluing matching-shaped tables together.

---

**Exercises for this lecture** (Lab, Exit Ticket, Optional Challenge) live in a separate notebook: `lect04_datetime_cleaning_reshaping_exercise.ipynb`.

---

## Part 4 — Cheat Sheet

| Task | Pandas |
|---|---|
| Parse text into a real datetime | `pd.to_datetime(col)` |
| Pull out a date component | `col.dt.year`, `.dt.month`, `.dt.day_name()`, `.dt.hour` |
| Compute a duration | `(col2 - col1).dt.total_seconds()` |
| Count missing values | `df.isna().sum()` |
| Find/drop duplicate rows | `df.duplicated()` / `df.drop_duplicates()` |
| Normalize inconsistent text | `.str.strip()`, `.str.upper()` / `.str.lower()` |
| Fix a wrongly-typed column | `pd.to_numeric(col, errors="coerce")` |
| Wide table from long | `df.pivot_table(index=..., columns=..., values=..., aggfunc=...)` |
| Long table from wide | `df.melt(id_vars=..., var_name=..., value_name=...)` |
| Stack tables vertically | `pd.concat([df1, df2])` |
| Combine tables sideways (Lecture 3) | `df1.merge(df2, on=...)` |

---

## Part 5 — Key Terms

- **`datetime64`**: Pandas' actual date/time data type, as opposed to text that merely looks like a date.
- **`.dt` accessor**: the gateway to date-specific operations (`.dt.year`, `.dt.day_name()`, etc.) on a datetime column.
- **Timedelta**: the result of subtracting one datetime from another — a duration, not a date.
- **Missing value**: an entry with no data (`NaN` for numbers, `NaT` for datetimes) — not always random, and not always something to fill in.
- **Duplicate row**: an exact repeat of another row in the same table.
- **Wide vs. long format**: wide has one column per category; long has one row per (category, value) pair.
- **`pivot_table` / `melt`**: reshape a table from long to wide, or wide to long.
- **`concat`**: stack tables with matching columns on top of each other.